# Storing Data with SQLite3

本笔记演示如何将金融时间序列 `DataFrame` 写入 SQLite3，并通过 SQL 查询执行简单分析。

- 生成样本时间序列数据
- 使用 `to_sql()` 写入 SQLite3
- 运行 SQL 过滤查询
- 查看查询结果与结果数量
- 关闭连接并可选清理文件

In [1]:
import sqlite3 as sq3
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 120)

In [2]:
def generate_sample_data(rows=10_000, cols=5, seed=42):
    """Generate a reproducible financial-like DataFrame."""
    rng = np.random.default_rng(seed)
    index = pd.date_range("2021-01-01", periods=rows, freq="min")

    steps = rng.normal(loc=0.0, scale=0.02, size=(rows, cols))
    data = 100 + np.cumsum(steps, axis=0)
    columns = [f"No{i}" for i in range(cols)]
    return pd.DataFrame(data, index=index, columns=columns).round(4)

## 1) 生成金融时间序列样本数据

说明：
- 为了保证演示在普通机器上更稳，默认 `ROWS = 200_000`。
- 如果你想复现书中的规模，可改成 `ROWS = 1_000_000`。

In [3]:
ROWS = 10_000
COLS = 5

t0 = perf_counter()
data = generate_sample_data(ROWS, COLS).round(4)
generate_seconds = perf_counter() - t0

print(f"Generated data in {generate_seconds:.3f} s")
data.info()
data.head()

Generated data in 0.009 s
<class 'pandas.DataFrame'>
DatetimeIndex: 10000 entries, 2021-01-01 00:00:00 to 2021-01-07 22:39:00
Freq: min
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   No0     10000 non-null  float64
 1   No1     10000 non-null  float64
 2   No2     10000 non-null  float64
 3   No3     10000 non-null  float64
 4   No4     10000 non-null  float64
dtypes: float64(5)
memory usage: 468.8 KB


,No0,No1,No2,No3,No4
2021-01-01 00:00:00,100.0061,99.9792,100.0150,100.0188,99.9610
2021-01-01 00:01:00,99.9801,99.9818,100.0087,100.0185,99.9439
2021-01-01 00:02:00,99.9976,99.9973,100.0100,100.0410,99.9533
2021-01-01 00:03:00,99.9805,100.0047,99.9908,100.0586,99.9523
2021-01-01 00:04:00,99.9768,99.9911,100.0153,100.0555,99.9437


## 2) 写入 SQLite3 数据库

这里使用 `DataFrame.to_sql()` 写入表 `data`，并统计写入耗时与数据库文件大小。

In [5]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
db_path = data_dir / "data.sqlite"

con = sq3.connect(db_path)

t0 = perf_counter()
data.to_sql("data", con, if_exists="replace", index=True, index_label="timestamp")
write_seconds = perf_counter() - t0

file_size_mb = db_path.stat().st_size / (1024 ** 2)
print(f"SQLite write time: {write_seconds:.3f} s")
print(f"Database file: {db_path}")
print(f"Database size: {file_size_mb:.2f} MB")

SQLite write time: 0.084 s
Database file: data/data.sqlite
Database size: 0.99 MB


## 3) 使用 SQL 查询筛选结果

下面执行示例查询：选取 `No1 > 100` 且 `No2 < 101` 的所有记录。

In [8]:
query = "SELECT * FROM data WHERE No1 > 100 and No2 < 101"

t0 = perf_counter()
res = con.execute(query).fetchall()
query_seconds = perf_counter() - t0

print(f"Query time: {query_seconds:.3f} s")
print(f"Number of rows: {len(res)}")

res[:5]

Query time: 0.004 s
Number of rows: 4988


[('2021-01-01 00:03:00', 99.9805, 100.0047, 99.9908, 100.0586, 99.9523),
 ('2021-01-01 00:05:00', 99.9697, 100.0017, 100.0226, 100.0638, 99.9523),
 ('2021-01-01 00:08:00', 100.05, 100.0022, 99.9822, 100.0356, 99.98),
 ('2021-01-01 00:09:00', 100.0544, 100.0196, 99.9867, 100.0492, 99.9813),
 ('2021-01-01 00:10:00', 100.0601, 100.0322, 99.9576, 100.0428, 99.9719)]

## 4) 关闭连接并可选清理

默认仅关闭连接，不删除数据库文件。若要删除生成文件，将 `CLEANUP = True`。

In [9]:
CLEANUP = False

con.close()
print("Connection closed.")

if CLEANUP and db_path.exists():
    db_path.unlink()
    print(f"Removed: {db_path}")
else:
    print(f"Kept: {db_path}")

Connection closed.
Kept: data/data.sqlite
